# 01. Auditoría de identidad de las imágenes: evidencia

Objetivo: encontrar fotografías de la misma mazorca o escena entre las 2211 imágenes analizadas y las 2076 sanas del repositorio que no forman parte del conjunto analizado. Requiere `dataT/` (ver `scripts/preparar_datos.py`) y el repositorio completo descomprimido en `dataset_complete/`.

Evidencia que se calcula para cada par candidato:

1. **Similitud DINOv2** (coseno entre descriptores ViT-S/14).
2. **Distancia pHash.**
3. **Distancia en la numeración** del repositorio (las fotos de una misma mazorca suelen tener números cercanos).
4. **Correspondencia geométrica SIFT + RANSAC**: número de puntos que encajan con una homografía. Dos fotos de la misma escena con un ligero cambio de encuadre comparten decenas o cientos de puntos; dos escenas distintas, casi ninguno.

Pares candidatos: todos los de la misma carpeta con números a distancia ≤ 30, los 10 vecinos DINOv2 más parecidos de cada imagen y todos los pares con coseno ≥ 0.85.

Este notebook solo calcula la evidencia y la guarda en `auditoria_identidad/`; la regla de agrupación y la revisión manual están en el 02. Duración estimada: 10 a 20 minutos (la mayor parte es SIFT en CPU).


## 1. Imágenes, MD5, pHash y copias de 256 px

In [ ]:
import os, re, json, time, glob, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image
import imagehash
import torch, timm, cv2
import torchvision.transforms as T
from timm.data import resolve_data_config
from IPython.display import display

PROJECT_DIR = Path.cwd()                                   # raíz del repositorio
CSV_PATH = PROJECT_DIR / 'dataT' / 'data.csv'              # 2211 imágenes analizadas (ver scripts/preparar_datos.py)
REPO_DIR = PROJECT_DIR / 'dataset_complete' / 'cacao_diseases' / 'cacao_photos'   # repositorio completo descomprimido
CACHE_DIR = PROJECT_DIR / 'cache_256'                      # copias RGB de 256 × 256 px (bicúbica)
OUT = PROJECT_DIR / 'auditoria_identidad'
OUT.mkdir(exist_ok=True)
DEV = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
HILOS = min(32, os.cpu_count() or 8)
cv2.setNumThreads(1)
VENTANA_NUM = 30          # pares de la misma carpeta con números a distancia <= 30
K_VECINOS = 10            # vecinos DINOv2 por imagen
COS_MIN = 0.85            # y todos los pares con coseno >= 0.85
LADO_SIFT = 640           # las imágenes originales se reducen a 640 × 640 para SIFT
print('dispositivo', DEV, '| hilos CPU', HILOS, '| OpenCV', cv2.__version__)

def md5(p):
    with open(p, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()

# Conjunto principal (2211) y sanas del repositorio que no forman parte de él (identificadas por MD5)
df = pd.read_csv(CSV_PATH)
a = pd.DataFrame({'image_path': df['image_path'], 'label': df['status'].astype(int), 'conjunto': 'principal'})
sanas_repo = sorted(glob.glob(str(REPO_DIR / 'healthy' / '*.jpg')))
with ThreadPoolExecutor(16) as ex:
    md5_a = list(ex.map(md5, [str(PROJECT_DIR / p) for p in a['image_path']]))
    md5_r = list(ex.map(md5, sanas_repo))
usadas = set(md5_a)
b = pd.DataFrame({'image_path': [str(Path(p).relative_to(PROJECT_DIR)) for p, h in zip(sanas_repo, md5_r) if h not in usadas],
                  'label': 0, 'conjunto': 'no_usada'})
img = pd.concat([a, b], ignore_index=True)
img['md5'] = md5_a + [h for h in md5_r if h not in usadas]
assert img['md5'].is_unique

# Mismo preprocesamiento que el entrenamiento: RGB, copia de 256 px (bicúbica); pHash y tamaño original
def procesar(rel):
    src = PROJECT_DIR / rel
    dst = CACHE_DIR / (os.path.splitext(rel)[0] + '.png')
    dst.parent.mkdir(parents=True, exist_ok=True)
    im = Image.open(src).convert('RGB')
    w, h = im.size
    ph = str(imagehash.phash(im))
    if not dst.exists():
        im.resize((256, 256), Image.Resampling.BICUBIC).save(dst)
    return ph, w, h, str(dst.relative_to(PROJECT_DIR))

with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(procesar, img['image_path']))
img['phash'], img['w'], img['h'], img['cache_path'] = zip(*res)
img['fuente'] = np.where(img['w'] == 1080, 'A', np.where(img['w'] == 2160, 'B', 'otra'))   # estratos de resolución
img['carpeta'] = np.where(img['label'] == 1, 'bpr', 'sana')     # espacio de numeración del repositorio
img['num'] = img['image_path'].str.extract(r'_(\d+)\.\w+$')[0].astype(int)
img.to_csv(OUT / 'imagenes.csv', index_label='id')
print(img.groupby(['conjunto', 'carpeta']).size())


## 2. DINOv2 y pares candidatos

In [ ]:
# 1) Descriptores DINOv2 ViT-S/14 sobre las copias de 256 px redimensionadas a 224 px
fm = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0, img_size=224).eval().to(DEV)
dc = resolve_data_config({}, model=fm)
tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(dc['mean'], dc['std'])])
feats = []
with torch.no_grad():
    for i in range(0, len(img), 128):
        xb = torch.stack([tf(Image.open(p).convert('RGB')) for p in img['cache_path'].iloc[i:i + 128].apply(lambda r: str(PROJECT_DIR / r))]).to(DEV)
        feats.append(torch.nn.functional.normalize(fm(xb).float(), dim=1).cpu())
        print('\rDINOv2 %d/%d' % (min(i + 128, len(img)), len(img)), end='', flush=True)
feats = torch.cat(feats).numpy().astype(np.float32)
del fm
np.save(OUT / 'feats_dinov2.npy', feats)
cos = feats @ feats.T
np.fill_diagonal(cos, -1)
print('\nlisto')

# 2) Pares candidatos
n = len(img)
cand = set()
num, carp = img['num'].values, img['carpeta'].values
orden = np.lexsort((num, carp))
for pos, i in enumerate(orden):                                   # (a) números cercanos en la misma carpeta
    for j in orden[pos + 1:]:
        if carp[j] != carp[i] or num[j] - num[i] > VENTANA_NUM:
            break
        cand.add((min(i, j), max(i, j)))
vec = np.argsort(-cos, axis=1)[:, :K_VECINOS]                      # (b) k vecinos más parecidos
for i in range(n):
    for j in vec[i]:
        cand.add((min(i, j), max(i, j)))
ii, jj = np.where(np.triu(cos, 1) >= COS_MIN)                     # (c) coseno alto
cand |= set(zip(ii.tolist(), jj.tolist()))
pares = pd.DataFrame(sorted(cand), columns=['i', 'j'])
pares['cos'] = cos[pares['i'], pares['j']]
h = np.array([int(x, 16) for x in img['phash']], dtype=np.uint64)
pares['ham'] = [bin(int(h[i]) ^ int(h[j])).count('1') for i, j in zip(pares['i'], pares['j'])]
pares['misma_carpeta'] = carp[pares['i']] == carp[pares['j']]
pares['dnum'] = np.where(pares['misma_carpeta'], np.abs(num[pares['i']] - num[pares['j']]), -1)
pares['rango_vecino'] = [min(int(np.where(vec[i] == j)[0][0]) if j in vec[i] else 99,
                             int(np.where(vec[j] == i)[0][0]) if i in vec[j] else 99) for i, j in zip(pares['i'], pares['j'])]
print('Pares candidatos:', len(pares))


## 3. SIFT + RANSAC y resumen

In [ ]:
# 3) SIFT en las imágenes originales reducidas a 640 px
sift_cfg = dict(nfeatures=1500)
KP, DS = [None] * n, [None] * n
def extraer(i):
    g = Image.open(PROJECT_DIR / img['image_path'].iat[i]).convert('L').resize((LADO_SIFT, LADO_SIFT), Image.Resampling.BICUBIC)
    k, d = cv2.SIFT_create(**sift_cfg).detectAndCompute(np.asarray(g), None)
    KP[i] = np.float32([p.pt for p in k]) if k else np.zeros((0, 2), np.float32)
    DS[i] = d if d is not None else np.zeros((0, 128), np.float32)
t0 = time.time()
with ThreadPoolExecutor(HILOS) as ex:
    for c, _ in enumerate(ex.map(extraer, range(n)), 1):
        if c % 250 == 0 or c == n:
            print('\rSIFT: %d/%d imágenes | %.0f s' % (c, n, time.time() - t0), end='', flush=True)
print()

# 4) Emparejamiento con prueba de razón (0.75) y RANSAC (homografía, 8 px)
def emparejar(par):
    i, j = par
    da, db = DS[i], DS[j]
    if len(da) < 8 or len(db) < 8:
        return 0, 0
    m = cv2.FlannBasedMatcher(dict(algorithm=1, trees=4), dict(checks=64)).knnMatch(da, db, k=2)
    g = [p[0] for p in m if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(g) < 8:
        return len(g), 0
    pa = KP[i][[x.queryIdx for x in g]]
    pb = KP[j][[x.trainIdx for x in g]]
    _, mask = cv2.findHomography(pa, pb, cv2.RANSAC, 8.0)
    return len(g), int(mask.sum()) if mask is not None else 0

lista = list(zip(pares['i'].tolist(), pares['j'].tolist()))
res = [None] * len(lista)
t0 = time.time()
with ThreadPoolExecutor(HILOS) as ex:
    for c, r in enumerate(ex.map(emparejar, lista, chunksize=64)):
        res[c] = r
        if (c + 1) % 2000 == 0 or c + 1 == len(lista):
            v = (c + 1) / (time.time() - t0)
            print('\rRANSAC: %d/%d pares | %.0f pares/s | resta ~%.0f s' % (c + 1, len(lista), v, (len(lista) - c - 1) / v),
                  end='', flush=True)
print()
pares['n_corresp'], pares['inliers'] = zip(*res)
pares.to_csv(OUT / 'pares_candidatos.csv', index=False)

# 5) Resumen rápido (para decidir la regla de agrupación)
pares['banda_cos'] = pd.cut(pares['cos'], [-1, 0.80, 0.85, 0.90, 0.93, 0.95, 1.01])
pares['banda_inl'] = pd.cut(pares['inliers'], [-1, 14, 39, 99, 10**6], labels=['<15', '15-39', '40-99', '>=100'])
display(pd.crosstab(pares['banda_cos'], pares['banda_inl'], margins=True))
cerc = pares[pares['misma_carpeta']]
cerc = cerc.assign(banda_dnum=pd.cut(cerc['dnum'], [0, 2, 5, 10, 20, 30, 10**6]))
display(pd.crosstab(cerc['banda_dnum'], cerc['banda_inl'], margins=True))
print('Guardado en', OUT)
